# Pandas: Raw DataFrames

So far we've loaded data from neatly formatted files. In the real world, data often arrives **raw**: as a Python dictionary, as a NumPy array, as a headerless CSV, or with missing values scattered all over.

This lesson covers three practical workflows:

1. **Building a DataFrame from a Python dictionary**.
2. **Building a DataFrame from NumPy arrays** — and seeing how NumPy and pandas work together.
3. **Loading and cleaning a messy DataFrame** (set column names, handle `NaN`, drop columns/rows, reset the index).


In [ ]:
import pandas as pd

## Part 1 — Build a DataFrame from a Dictionary

A `DataFrame` can be constructed directly from a Python `dict`, where:

- each **key** becomes a **column name**
- each **value** is a list/array of values for that column

This is the fastest way to create small, hand-crafted datasets — useful for demos, tests, and quick experiments.

In [ ]:
raw_data = {
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'age': [25, 30, 35, 28],
    'city': ['Cairo', 'Riyadh', 'Dubai', 'Amman'],
    'active': [True, False, True, True],
}

df = pd.DataFrame(raw_data)
df

### Reorder columns

By default, columns appear in the order the dictionary keys were inserted. To present them in a specific order, **index the DataFrame with a list of column names**:

In [ ]:
df = df[['name', 'city', 'age', 'active']]
df

### Add a new column

Assigning a list (or `Series`) to a new column name creates the column:

In [ ]:
df['country'] = ['Egypt', 'Saudi Arabia', 'UAE', 'Jordan']
df

### Inspect column types

Pandas **infers** a dtype for each column based on its values. Check them with `.dtypes`:

In [ ]:
df.dtypes

Strings show up as `object`, integers as `int64`, booleans as `bool`, and so on.

## Part 2 — Build a DataFrame from NumPy Arrays

Pandas is built **on top of NumPy**. Under the hood, every numeric DataFrame column is stored as a NumPy array, and most of pandas' fast numerical operations are powered by NumPy.

A useful way to think about the split:

- **NumPy** provides the `ndarray` — a fast, homogeneous (single-dtype), unlabeled n-dimensional array. It's the *math engine*.
- **Pandas** wraps NumPy arrays with **labeled rows and columns**, **mixed dtypes**, and table-aware operations (joins, group-by, time series, I/O). It's the *table layer*.

The two are designed to interoperate: you can hand a NumPy array to `pd.DataFrame`, and you can pull a NumPy array back out of any DataFrame.

In [ ]:
import numpy as np

arr = np.array([
    [5.1, 3.5, 1.4],
    [4.9, 3.0, 1.4],
    [4.7, 3.2, 1.3],
    [4.6, 3.1, 1.5],
])

pd.DataFrame(arr)

Without labels, pandas falls back to **integer column names** (`0, 1, 2, ...`) and an integer index. Provide your own labels with the `columns=` (and optionally `index=`) arguments:

In [ ]:
df_np = pd.DataFrame(
    arr,
    columns=['sepal_length', 'sepal_width', 'petal_length'],
    index=['a', 'b', 'c', 'd'],
)
df_np

### Generate synthetic data with NumPy

NumPy shines at generating numerical data — `np.arange`, `np.linspace`, random sampling, etc. You can drop those arrays straight into a `pd.DataFrame`. This is the standard way to build **synthetic datasets** for testing, plotting, or teaching:

In [ ]:
rng = np.random.default_rng(seed=42)

df_synthetic = pd.DataFrame({
    'id':       np.arange(5),
    'price':    rng.uniform(10, 100, size=5).round(2),
    'in_stock': rng.integers(0, 2, size=5).astype(bool),
})
df_synthetic

Notice how each column can have its **own dtype** (`int64`, `float64`, `bool`) — that's the pandas part. NumPy alone forces a single dtype across the whole array.

### NumPy functions work on DataFrames

Because pandas stores its data as NumPy arrays, **NumPy functions just work** on Series and DataFrames — they operate elementwise and return a pandas object back:

In [ ]:
np.log(df_synthetic['price'])

### Going back to NumPy

Any DataFrame can be unwrapped into a raw NumPy array with `.to_numpy()`. This is the right move when you need to hand data to a library that expects a plain matrix — like **scikit-learn**, **SciPy**, or many plotting functions:

In [ ]:
df_np.to_numpy()

> Labels (index and column names) are **dropped** when you go back to NumPy — only the raw values remain.

### How NumPy and pandas work together — mental model

| Layer  | What it does                                      | Typical use                          |
| ------ | ------------------------------------------------- | ------------------------------------ |
| NumPy  | Fast math on raw `ndarray`s (one dtype, no labels) | Vectorized computation, linear algebra |
| Pandas | Labeled tables on top of NumPy (mixed dtypes)     | Cleaning, exploration, joins, I/O    |

In practice you move freely between them:

1. **Load and clean** raw data with pandas (`read_csv`, `dropna`, `fillna`, ...).
2. **Hand the underlying array** to NumPy / scikit-learn / SciPy with `.to_numpy()` for heavy numerical work.
3. **Wrap the result** back into a DataFrame for inspection, plotting, or further analysis.

## Part 3 — Load and Clean a Raw CSV

Real datasets are rarely tidy. Many CSV files come with **no header row**, and once loaded they may contain **missing values** (`NaN`) you have to deal with.

We'll use the classic **Iris dataset** to walk through:

- reading a CSV that has no header
- naming the columns ourselves
- detecting and filling missing values
- dropping columns and rows
- resetting the index

In [ ]:
import numpy as np

### Read a CSV with no header and set column names

By default, `pd.read_csv` treats the first row as the header.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'

iris = pd.read_csv(url)
iris.head(3)

When the file has **no header**, pass `header=None` and provide the column names yourself with `names=[...]`:

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'

iris = pd.read_csv(
    url,
    header=None,
    names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class'],
)
iris.head()

### Detect missing values

`isna()` (alias `isnull()`) returns a boolean DataFrame of the same shape, with `True` wherever a value is `NaN`. Combine it with `.sum()` to count missing values **per column**, or `.any().any()` to ask the global question *"are there any missing values at all?"*:

In [ ]:
iris.isna().sum()

The dataset is clean — no `NaN` values yet. Let's **introduce some**, so we can practice cleaning them.

### Set specific cells to NaN

Use `.loc[rows, columns]` to assign `np.nan` to a slice of the DataFrame. Here we corrupt rows `10` through `29` of the `petal_length` column:

In [ ]:
iris.loc[10:29, 'petal_length'] = np.nan
iris.isna().sum()

> Note: with `.loc`, the **end label is inclusive** — `10:29` covers 20 rows (10, 11, …, 29).

### Fill missing values

`fillna(value)` replaces every `NaN` with `value`. You can fill with a constant, the column **mean**, **median**, the previous value (`method='ffill'`), etc. Here we fill with `1.0`:

In [ ]:
iris['petal_length'] = iris['petal_length'].fillna(1.0)
iris.isna().sum()

### Drop a column

Use `df.drop(columns=[...])` to remove one or more columns. By default this returns a **new** DataFrame, so you must reassign (or pass `inplace=True`):

In [ ]:
iris = iris.drop(columns=['class'])
iris.head()

### Drop rows containing NaN

Let's deliberately set the **first 3 rows** to `NaN`:

In [ ]:
iris.iloc[0:3, :] = np.nan
iris.head()

Now use `dropna()` to remove **every row that contains at least one `NaN`**:

In [ ]:
iris = iris.dropna()
iris.head()

### Reset the index

After dropping rows, the **index becomes non-contiguous** (notice it starts at `3` now, not `0`). `reset_index(drop=True)` renumbers the index starting from `0`, throwing away the old one:

In [ ]:
iris = iris.reset_index(drop=True)
iris.head()

> Without `drop=True`, the old index is kept as a new column called `index`.

## Key Takeaways

**Building DataFrames from a dictionary**

- `pd.DataFrame(dict)` — each key becomes a column.
- Reorder columns with `df[['col_a', 'col_b', ...]]`.
- Add a new column by assignment: `df['new_col'] = [...]`.
- Inspect types with `df.dtypes`.

**Building DataFrames from NumPy (and how they work together)**

- Pandas is built **on top of NumPy** — numeric columns are NumPy arrays under the hood.
- `pd.DataFrame(np_array, columns=[...], index=[...])` — wrap a NumPy array as a labeled table.
- A 2-D NumPy array has a **single dtype**; a DataFrame can mix dtypes across columns.
- NumPy generators (`np.arange`, `np.linspace`, `np.random.default_rng(...)`) are the standard way to create synthetic data for DataFrames.
- NumPy functions (`np.log`, `np.where`, `np.mean`, ...) work directly on Series and DataFrames.
- `df.to_numpy()` — drop the labels and get a raw NumPy array (e.g. to feed into scikit-learn).

**Loading and cleaning raw data**

- `pd.read_csv(url, header=None, names=[...])` — read a headerless CSV and name the columns yourself.
- `df.isna().sum()` — count missing values per column.
- `df.loc[rows, cols] = np.nan` — assign `NaN` to specific cells.
- `df['col'].fillna(value)` — replace `NaN` with a value (constant, mean, median, …).
- `df.drop(columns=[...])` — remove columns.
- `df.dropna()` — drop rows containing `NaN`.
- `df.reset_index(drop=True)` — renumber the index after dropping rows.